# Ensemble Boilerplate

Runs the `src/ensemble` voting utilities on saved binary and multiclass model files.

- Binary models: `models/binary/*.joblib` or `models/binary/*.pkl`
- Multiclass models: `models/multi/*.joblib` or `models/multi/*.pkl`

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score

pd.set_option('display.max_columns', None)

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
ENSEMBLE_SRC_DIR = SRC_DIR / 'ensemble'
BINARY_MODEL_DIR = PROJECT_ROOT / 'models' / 'binary'
MULTI_MODEL_DIR = PROJECT_ROOT / 'models' / 'multi'
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

# ensemble.py imports sibling modules as top-level imports.
for path in [SRC_DIR, ENSEMBLE_SRC_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from ensemble import Ensemble

TEXT_COL = 'message'
LABEL_COL = 'label'
SEED = 7524

BINARY_MODEL_DIR.mkdir(parents=True, exist_ok=True)
MULTI_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Binary model dir:', BINARY_MODEL_DIR)
print('Multiclass model dir:', MULTI_MODEL_DIR)

In [ ]:
def load_model_files(model_dir):
    model_paths = sorted(model_dir.glob('*.joblib')) + sorted(model_dir.glob('*.pkl'))
    models = []

    for path in model_paths:
        model = joblib.load(path)
        models.append(model)

    return model_paths, models


def score_predictions(y_true, y_pred, average):
    return {
        'f1': round(float(f1_score(y_true, y_pred, average=average, zero_division=0)), 4),
        'recall': round(float(recall_score(y_true, y_pred, average=average, zero_division=0)), 4),
        'precision': round(float(precision_score(y_true, y_pred, average=average, zero_division=0)), 4),
        'accuracy': round(float(accuracy_score(y_true, y_pred)), 4),
    }


def score_func_for_average(average):
    return lambda y_true, y_pred: f1_score(y_true, y_pred, average=average, zero_division=0)


def evaluate_base_models(models, model_paths, X, y, average):
    rows = []
    if not models:
        return pd.DataFrame([{
            'model': None,
            'f1': np.nan,
            'recall': np.nan,
            'precision': np.nan,
            'accuracy': np.nan,
            'supports_predict_proba': None,
            'skipped_reason': 'No model files found.',
        }])

    for path, model in zip(model_paths, models):
        y_pred = model.predict(X)
        rows.append({
            'model': path.name,
            **score_predictions(y, y_pred, average),
            'supports_predict_proba': hasattr(model, 'predict_proba'),
        })

    return pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)


def evaluate_ensemble_methods(models, X, y, average, n_weight_trials=1000):
    if not models:
        return None, pd.DataFrame([{
            'ensemble': None,
            'f1': np.nan,
            'recall': np.nan,
            'precision': np.nan,
            'accuracy': np.nan,
            'skipped_reason': 'No model files found.',
        }])

    ensemble = Ensemble(models)
    score_func = score_func_for_average(average)
    rows = []

    simple_pred = ensemble.predict_simple_majority(X)
    rows.append({'ensemble': 'simple_majority', **score_predictions(y, simple_pred, average)})

    weighted_weights, weighted_score = ensemble.fit_weighted_majority(
        X_val=X,
        y_val=y,
        score_func=score_func,
        n_trials=n_weight_trials,
        random_state=SEED,
    )
    weighted_pred = ensemble.predict_weighted_majority(X, weighted_weights)
    rows.append({
        'ensemble': 'weighted_majority',
        **score_predictions(y, weighted_pred, average),
        'fit_score': round(float(weighted_score), 4),
        'weights': np.round(weighted_weights, 4).tolist(),
    })

    if all(hasattr(model, 'predict_proba') for model in models):
        confidence_weights, confidence_score = ensemble.fit_weighted_confidence_majority(
            X_val=X,
            y_val=y,
            score_func=score_func,
            n_trials=n_weight_trials,
            random_state=SEED,
        )
        confidence_pred = ensemble.predict_weighted_confidence_majority(X, confidence_weights)
        rows.append({
            'ensemble': 'weighted_confidence_majority',
            **score_predictions(y, confidence_pred, average),
            'fit_score': round(float(confidence_score), 4),
            'weights': np.round(confidence_weights, 4).tolist(),
        })
    else:
        rows.append({
            'ensemble': 'weighted_confidence_majority',
            'skipped_reason': 'At least one model does not support predict_proba.',
        })

    return ensemble, pd.DataFrame(rows).sort_values('f1', ascending=False, na_position='last').reset_index(drop=True)


def fit_and_score_stacked(models, X_train, y_train, X_eval, y_eval, average):
    meta_model = LogisticRegression(max_iter=1000, random_state=SEED)
    ensemble = Ensemble(models)
    ensemble.fit_stacked(X_train, y_train, meta_model=meta_model, n_folds=5, random_state=SEED)
    y_pred = ensemble.predict_stacked(X_eval)
    return ensemble, score_predictions(y_eval, y_pred, average)

In [ ]:
# Validation/test split used for ensemble comparison.





## Binary Ensemble

Uses models saved under `models/binary`. Labels are converted to two classes: `0 = non-toxic`, `1 = toxic`.

In [ ]:
binary_model_paths, binary_models = load_model_files(BINARY_MODEL_DIR)
print(f'Loaded {len(binary_models)} binary models')
for path in binary_model_paths:
    print('-', path.name)

In [ ]:
binary_base_scores = evaluate_base_models(
    binary_models,
    binary_model_paths,
    X_val,
    y_val_binary,
    average='binary',
)
binary_base_scores

In [ ]:
binary_ensemble, binary_ensemble_scores = evaluate_ensemble_methods(
    binary_models,
    X_val,
    y_val_binary,
    average='binary',
    n_weight_trials=1000,
)
binary_ensemble_scores

In [ ]:
# Final binary ensemble report on test using the best validation ensemble by F1.
if binary_ensemble is None:
    print('No binary ensemble was evaluated because no binary model files were found.')
else:
    best_binary_method = binary_ensemble_scores.iloc[0]['ensemble']

    if best_binary_method == 'simple_majority':
        y_binary_test_pred = binary_ensemble.predict_simple_majority(X_test)
    elif best_binary_method == 'weighted_majority':
        y_binary_test_pred = binary_ensemble.predict_weighted_majority(X_test)
    elif best_binary_method == 'weighted_confidence_majority':
        y_binary_test_pred = binary_ensemble.predict_weighted_confidence_majority(X_test)
    else:
        raise ValueError(f'Unsupported ensemble method: {best_binary_method}')

    print('Best binary ensemble:', best_binary_method)
    print(score_predictions(y_test_binary, y_binary_test_pred, average='binary'))
    print(classification_report(y_test_binary, y_binary_test_pred, zero_division=0))

## Multiclass Ensemble

Uses models saved under `models/multi`. Labels stay in the original multiclass scheme.

In [ ]:
multi_model_paths, multi_models = load_model_files(MULTI_MODEL_DIR)
print(f'Loaded {len(multi_models)} multiclass models')
for path in multi_model_paths:
    print('-', path.name)

In [ ]:
multi_base_scores = evaluate_base_models(
    multi_models,
    multi_model_paths,
    X_val,
    y_val_multi,
    average='macro',
)
multi_base_scores

In [ ]:
multi_ensemble, multi_ensemble_scores = evaluate_ensemble_methods(
    multi_models,
    X_val,
    y_val_multi,
    average='macro',
    n_weight_trials=1000,
)
multi_ensemble_scores

In [ ]:
# Final multiclass ensemble report on test using the best validation ensemble by macro F1.
if multi_ensemble is None:
    print('No multiclass ensemble was evaluated because no multiclass model files were found.')
else:
    best_multi_method = multi_ensemble_scores.iloc[0]['ensemble']

    if best_multi_method == 'simple_majority':
        y_multi_test_pred = multi_ensemble.predict_simple_majority(X_test)
    elif best_multi_method == 'weighted_majority':
        y_multi_test_pred = multi_ensemble.predict_weighted_majority(X_test)
    elif best_multi_method == 'weighted_confidence_majority':
        y_multi_test_pred = multi_ensemble.predict_weighted_confidence_majority(X_test)
    else:
        raise ValueError(f'Unsupported ensemble method: {best_multi_method}')

    print('Best multiclass ensemble:', best_multi_method)
    print(score_predictions(y_test_multi, y_multi_test_pred, average='macro'))
    print(classification_report(y_test_multi, y_multi_test_pred, zero_division=0))

## Optional Stacking

`src/ensemble/stacked.py` retrains clones of the loaded base model objects, so only run this section for sklearn-compatible pipelines/models that can be cloned and fitted.

In [ ]:
# Binary stacking example
# binary_stacked_ensemble, binary_stacked_scores = fit_and_score_stacked(
#     binary_models,
#     X_val,
#     y_val_binary,
#     X_test,
#     y_test_binary,
#     average='binary',
# )
# binary_stacked_scores

# Multiclass stacking example
# multi_stacked_ensemble, multi_stacked_scores = fit_and_score_stacked(
#     multi_models,
#     X_val,
#     y_val_multi,
#     X_test,
#     y_test_multi,
#     average='macro',
# )
# multi_stacked_scores